Author: Krish

In [1]:
# Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from pathlib import Path
from functools import reduce

### User input required
Put the data path on your system in the cell below

In [2]:
# Enter data path
data_path = "C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Spring 2025\\STAT390\\LegalAid\\Data\\All Calls by Month\\"

### User input ends

### Reading all filenames in the data folder

In [3]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

### Reading first 5 rows of all data files
The code chunk below reads the first 5 rows of all data files. This is to check the columns that are present in all the data files.

In [4]:
i=0; df = []
for f in files:
    if f.suffix.lower() == ".csv":
        df.append(pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False, nrows = 5))
    else:  # .xlsx
        df.append(pd.read_excel(f, sheet_name=0, header=0, dtype=str, nrows = 5))
    #df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df[i].shape)
    i = i + 1

0 April 2024 (5, 63)
1 April 2025 (5, 63)
2 August 2024 (5, 57)
3 August 2025 (5, 63)
4 December 2024 (5, 55)
5 February 2025 (5, 63)
6 January 2025 (5, 64)
7 July 2024 (5, 55)
8 July 2025 (5, 63)
9 June 2024 (5, 63)
10 June 2025 (5, 63)
11 March 2025 (5, 66)
12 May 2024 (5, 63)
13 May 2025 (5, 63)
14 November 2024 (5, 63)
15 October 2024 (5, 55)
16 September 2024 (5, 55)
17 September 2025 (5, 69)


The code chunk below identifies the columns missing in at least one DataFrame.

In [5]:
all_cols = reduce(lambda x, y: x | set(y.columns), df[1:], set(df[0].columns))
common_cols = reduce(lambda x, y: x & set(y.columns), df[1:], set(df[0].columns))
common_cols
not_in_all = all_cols - common_cols
print("Columns missing from at least one dataframe:", not_in_all)

Columns missing from at least one dataframe: {'Public Called IP Address', 'Public Calling IP Address', 'Queue Type', 'Call Recording Result', 'Redirecting party UUID', 'User', 'Device owner UUID', 'Call Recording Trigger', 'Auto Attendant Key Pressed', 'Call Recording Platform Name', 'External caller ID number', 'Column1', 'PSTN vendor name2', 'Original reason2', 'Hold Duration', 'Original called party UUID', 'Recall Type', 'Answered Elsewhere'}


The code chunk below prints the columns present in all the data files.

In [6]:
print(common_cols)

{'Local call ID', 'Start time', 'Duration', 'Releasing party', 'Client version', 'Device Mac', 'Call type', 'Client type', 'Inbound trunk', 'Remote SessionID', 'Site timezone', 'Related reason', 'Answer time', 'PSTN vendor name', 'Transfer related call ID', 'Ring duration', 'PSTN provider ID', 'Model', 'Authorization code', 'PSTN legal entity', 'PSTN vendor Org ID', 'Org UUID', 'Final local sessionID', 'Outbound trunk', 'OS type', 'Answer Indicator', 'Called number', 'Site main number', 'Call outcome reason', 'Redirecting number', 'Sub client type', 'International Country', 'Redirect reason', 'User type', 'User number', 'Local SessionID', 'Report ID', 'Location', 'Direction', 'Call outcome', 'Network call ID', 'Release time', 'Original reason', 'Remote call ID', 'Call transfer time', 'User UUID', 'Correlation ID', 'Related call ID', 'Final remote sessionID', 'Route group', 'Site UUID', 'Report time', 'Department ID', 'Call ID', 'Answered'}


### Reading all the data files
All the datafiles are read with the common columns read first.

In [7]:
df_main = pd.DataFrame(columns=list(common_cols))

In [8]:
i=0; 
for f in files:
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=0, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=0, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
    print(i, f.stem, df.shape)
    i = i + 1

0 April 2024 (56662, 63)
1 April 2025 (63636, 63)
2 August 2024 (63262, 57)
3 August 2025 (57071, 63)
4 December 2024 (49445, 55)
5 February 2025 (63669, 63)
6 January 2025 (62623, 64)
7 July 2024 (62292, 55)
8 July 2025 (60438, 63)
9 June 2024 (56763, 63)
10 June 2025 (54598, 63)
11 March 2025 (59149, 66)
12 May 2024 (62944, 63)
13 May 2025 (55428, 63)
14 November 2024 (49953, 63)
15 October 2024 (62354, 55)
16 September 2024 (61250, 55)
17 September 2025 (56571, 69)


### Converting date to datetime format

In [9]:
df_main["Start time"] = pd.to_datetime(df_main["Start time"], utc=True).dt.tz_convert("America/Chicago").dt.tz_localize(None)

In [10]:
df_main.shape

(1058108, 73)

In [11]:
df_main['Correlation ID'].nunique()

516786

In [12]:
df_main.loc[df_main['Called number'] == '13123411070','Correlation ID'].nunique()

219766

In [13]:
df_main.loc[df_main['Called number'] == '13124312299','Correlation ID'].nunique()

10445

In [14]:
df_main.loc[df_main['Called number'] == '13123478347','Correlation ID'].nunique()

89

In [15]:
df_main.loc[df_main['Called number'] == '18882652188','Correlation ID'].nunique()

2

In [40]:
FD   = "13122296300"   # front desk number
MAIN = "13123411070"   # main number

df = df_main.copy()
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'[^0-9a-z]+', '_', regex=True)
)

# PARSE TIMES 
for col in ["start_time", "answer_time", "release_time", "report_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

# USING END TIMES
df["end_time"] = df["release_time"]
df["end_time"] = df["end_time"].fillna(df.get("report_time"))
df["end_time"] = df["end_time"].fillna(df.get("answer_time"))
df["end_time"] = df["end_time"].fillna(df.get("start_time"))

# ORDER LEGS WITHIN A CALL JOURNEY 
df = df.sort_values(["correlation_id", "end_time"])
df["leg_idx"]    = df.groupby("correlation_id").cumcount()
df["last_leg_ix"] = df.groupby("correlation_id")["leg_idx"].transform("max")
df["is_last_leg"] = df["leg_idx"].eq(df["last_leg_ix"])

In [50]:
df[['correlation_id', 'leg_idx', 'last_leg_ix','is_last_leg']].iloc[0:10,:]
# FRONT DESK SUBSETS
fd_inbound  = df[(df["called_number"] == FD) & (df["direction"] == "TERMINATING")]
fd_outbound = df[(df.get("user_number") == FD) & (df["direction"] == "ORIGINATING")]

# FRONT DESK AS DESTINATION (INBOUND)
print(f"Front desk appears as a call *destination* in {len(fd_inbound):,} legs.")
print("\nTop redirecting numbers → front desk:")
print(fd_inbound["redirecting_number"].value_counts(dropna=False).head(10))

Front desk appears as a call *destination* in 19,344 legs.

Top redirecting numbers → front desk:
redirecting_number
13123411070    16099
NaN             2515
13125068647      355
13125068646      173
13123478342      105
13127536357       41
13124312299       20
13125068649       16
13127536356       14
13122296080        4
Name: count, dtype: int64


In [51]:
#FRONT DESK AS CALLER (OUTBOUND)
print(f"\nFront desk appears as *caller* in {len(fd_outbound):,} legs.")
print("Top numbers the front desk calls:")
print(fd_outbound["called_number"].value_counts(dropna=False).head(10))


Front desk appears as *caller* in 13,718 legs.
Top numbers the front desk calls:
called_number
1180           5836
1182           2132
13123478300    1611
13122296346     314
13125068649     218
13123478375     157
13123478326     157
13123478312     131
13127221813     105
13122296341      88
Name: count, dtype: int64


In [52]:
#IS THE FRONT DESK THE FINAL LEG (USING END_TIME)?
fd_total_journeys = df.loc[df["called_number"] == FD, "correlation_id"].nunique()
fd_last_journeys  = df.loc[(df["called_number"] == FD) & (df["is_last_leg"]), "correlation_id"].nunique()
fd_last_pct       = (fd_last_journeys / fd_total_journeys * 100) if fd_total_journeys else 0

In [53]:
print(f"\nFront desk is the final leg in {fd_last_journeys:,} of {fd_total_journeys:,} call journeys "
      f"({fd_last_pct:.2f}%).")


Front desk is the final leg in 11,894 of 18,595 call journeys (63.96%).


In [54]:
# HOW OFTEN DOES MAIN → FRONT DESK OCCUR?
# Count FD inbound legs whose *last redirecting number* is MAIN
fd_inbound_from_main = fd_inbound.loc[fd_inbound["redirecting_number"] == MAIN]
legs_main_to_fd = len(fd_inbound_from_main)

# Count unique journeys where an FD inbound leg has redirecting_number == MAIN
journeys_main_to_fd = fd_inbound_from_main["correlation_id"].nunique()
print(f"\nLegs with MAIN → FD (by redirecting_number): {legs_main_to_fd:,}")
print(f"Journeys containing MAIN → FD: {journeys_main_to_fd:,}")


Legs with MAIN → FD (by redirecting_number): 16,099
Journeys containing MAIN → FD: 15,903


In [55]:
# WITHIN A JOURNEY, DOES FD EVER CALL BACK OUT TO MAIN? (FD → MAIN)
journeys_fd_to_main = df.groupby("correlation_id").apply(
    lambda g: ((g.get("user_number") == FD) & (g["called_number"] == MAIN)).any()
).sum()

print(f"Journeys containing FD → MAIN (FD placing a call to MAIN): {journeys_fd_to_main:,}")

Journeys containing FD → MAIN (FD placing a call to MAIN): 7


C:\Users\akl0407\AppData\Local\Temp\ipykernel_42580\274267280.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  journeys_fd_to_main = df.groupby("correlation_id").apply(


In [56]:
def next_hop_after_fd(group):
    # all FD legs in this journey
    fd_rows = group[group["called_number"] == FD]
    if fd_rows.empty:
        return None
    # take the last FD leg in this journey (by end_time order)
    last_fd_ix = fd_rows["leg_idx"].max()
    # the immediate next leg after the last FD leg (if any)
    candidate = group[group["leg_idx"] == last_fd_ix + 1]
    if candidate.empty:
        return None
    row = candidate.iloc[0]
    # return where it went next
    return f'{row.get("direction", "")}|to:{row.get("called_number", "")}|from:{row.get("user_number", "")}'

non_last_fd_journeys = df.loc[(df["called_number"] == FD) & (~df["is_last_leg"]), "correlation_id"].unique()
next_hops = (
    df[df["correlation_id"].isin(non_last_fd_journeys)]
      .groupby("correlation_id", group_keys=False)
      .apply(next_hop_after_fd)
)

C:\Users\akl0407\AppData\Local\Temp\ipykernel_42580\2754765154.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(next_hop_after_fd)


In [57]:
next_hops = next_hops.dropna()
print("\nTop immediate next hops after the last FD leg (for journeys where FD was not final):")
print(pd.Series(next_hops).value_counts().head(15))


Top immediate next hops after the last FD leg (for journeys where FD was not final):
TERMINATING|to:13123411070|from:13123411070    4497
TERMINATING|to:13125068646|from:13125068646     404
ORIGINATING|to:13123478300|from:13124235938     399
TERMINATING|to:13123478300|from:13123478300     377
TERMINATING|to:1180|from:1180                   287
TERMINATING|to:1182|from:1182                   140
TERMINATING|to:13125068647|from:13125068647     108
TERMINATING|to:13123478342|from:13123478342      31
TERMINATING|to:13122296346|from:13122296346      27
ORIGINATING|to:13123478300|from:13122296073      19
ORIGINATING|to:14154490512|from:13125068646      15
TERMINATING|to:13123478375|from:13123478375      14
ORIGINATING|to:13125068649|from:13124235938      13
ORIGINATING|to:13123478300|from:13122296346      12
ORIGINATING|to:2302|from:13123411070             12
Name: count, dtype: int64


In [38]:
df_main.to_csv('C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Processed_data\\all_calls_appended.csv',
              index = False)

In [2]:
df_main = pd.read_csv('C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Processed_data\\all_calls_appended.csv',
                     index_col = None)

C:\Users\akl0407\AppData\Local\Temp\ipykernel_35132\3962911233.py:1: DtypeWarning: Columns (10,11,36,50,56,60,64,65,66,67,70) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv('C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Processed_data\\all_calls_appended.csv',


In [6]:
df_main['Called number'].value_counts()

Called number
13123478300           235325
13123411070           222390
2302                   56716
13124235938            56152
13122296300            36376
                       ...  
13129098378                1
17736580657                1
#032235389800045#0         1
13122008607                1
12196144611                1
Name: count, Length: 48821, dtype: int64

In [9]:
df_main.loc[df_main['Called number'] == '13122296300','Correlation ID'].nunique()

18595

In [12]:
df_main.sort_values(by = ['Correlation ID','Start time'], inplace = True)

In [76]:
df_useful = df_main[['Correlation ID', 'Called number', 'PSTN vendor name', 'Direction', 'Duration', 'Original reason', 'Redirect reason',
        'Redirecting number', 'Start time', 'User type', 'User', 'Answered', 'Related reason', 'Release time', 'Answer time']].reset_index(drop = True)

In [78]:
df_useful.loc[df_useful['Correlation ID'] == '00006ccc-8250-4993-90c0-bbfd41f7dd24',:]	

,Correlation ID,Called number,PSTN vendor name,Direction,Duration,Original reason,Redirect reason,Redirecting number,Start time,User type,User,Answered,Related reason,Release time,Answer time
2,00006ccc-8250-4993-90c0-bbfd41f7dd24,13123478311,CallTower,TERMINATING,44,NaN,NaN,NaN,2024-12-11 08:41:47.355,User,NaN,True,NaN,2024-12-11T20:42:49.451Z,2024-12-11T20:42:05.392Z
3,00006ccc-8250-4993-90c0-bbfd41f7dd24,13123478300,NaN,ORIGINATING,44,NoAnswer,NoAnswer,13123478311.0,2024-12-11 08:42:05.358,User,NaN,True,CallForwardNoAnswer,2024-12-11T20:42:49.451Z,2024-12-11T20:42:05.392Z
4,00006ccc-8250-4993-90c0-bbfd41f7dd24,13123478300,NaN,TERMINATING,44,NoAnswer,NoAnswer,13123411070.0,2024-12-11 08:42:05.358,VoiceMailRetrieval,NaN,True,NaN,2024-12-11T20:42:49.451Z,2024-12-11T20:42:05.392Z


In [77]:
df_useful

,Correlation ID,Called number,PSTN vendor name,Direction,Duration,Original reason,Redirect reason,Redirecting number,Start time,User type,User,Answered,Related reason,Release time,Answer time
0,00000fe9-dfa0-41c5-986b-d9ce731f2715,13123411070,CallTower,TERMINATING,4,NaN,NaN,NaN,2025-06-27 06:25:23.791,WCCAdapter,NaN,True,NaN,2025-06-27T16:25:28.825Z,2025-06-27T16:25:23.921Z
1,00001e73-ce33-48f1-b531-bddc7ee3965d,13124312299,CallTower,TERMINATING,1961,NaN,NaN,NaN,2024-10-05 07:30:09.903,WCCAdapter,NaN,True,NaN,2024-10-05T18:02:52.100Z,2024-10-05T17:30:10.320Z
2,00006ccc-8250-4993-90c0-bbfd41f7dd24,13123478311,CallTower,TERMINATING,44,NaN,NaN,NaN,2024-12-11 08:41:47.355,User,NaN,True,NaN,2024-12-11T20:42:49.451Z,2024-12-11T20:42:05.392Z
3,00006ccc-8250-4993-90c0-bbfd41f7dd24,13123478300,NaN,ORIGINATING,44,NoAnswer,NoAnswer,13123478311.0,2024-12-11 08:42:05.358,User,NaN,True,CallForwardNoAnswer,2024-12-11T20:42:49.451Z,2024-12-11T20:42:05.392Z
4,00006ccc-8250-4993-90c0-bbfd41f7dd24,13123478300,NaN,TERMINATING,44,NoAnswer,NoAnswer,13123411070.0,2024-12-11 08:42:05.358,VoiceMailRetrieval,NaN,True,NaN,2024-12-11T20:42:49.451Z,2024-12-11T20:42:05.392Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1058103,ffffea69-f56b-4007-9136-97bcef7bb84e,2302,NaN,TERMINATING,153,FollowMe,FollowMe,13123411070,2025-07-22 07:42:53.228,AutomatedAttendantVideo,NaN,True,Deflection,2025-07-22T17:45:27.058Z,2025-07-22T17:42:53.265Z
1058104,ffffea69-f56b-4007-9136-97bcef7bb84e,13123478354,NaN,ORIGINATING,32,Deflection,Deflection,2302,2025-07-22 07:44:36.326,AutomatedAttendantVideo,NaN,True,Deflection,2025-07-22T17:45:27.058Z,2025-07-22T17:44:54.487Z
1058105,ffffea69-f56b-4007-9136-97bcef7bb84e,13123478354,NaN,TERMINATING,32,Deflection,Deflection,13125068649,2025-07-22 07:44:36.384,User,NaN,True,NaN,2025-07-22T17:45:27.058Z,2025-07-22T17:44:54.487Z
1058106,ffffea69-f56b-4007-9136-97bcef7bb84e,13123478300,NaN,ORIGINATING,32,Deflection,NoAnswer,13123478354,2025-07-22 07:44:54.386,User,NaN,True,CallForwardNoAnswer,2025-07-22T17:45:27.058Z,2025-07-22T17:44:54.487Z


In [62]:
df_frontdesk_rows = df_main.loc[df_main['Called number'] == '13122296300',:]

In [75]:
df_frontdesk_rows.loc[df_frontdesk_rows.Direction == 'TERMINATING']['User type'].value_counts()

User type
User    19344
Name: count, dtype: int64

In [63]:
df_frontdesk_rows['Redirect reason'].value_counts()

Redirect reason
FollowMe      33621
Deflection       40
Name: count, dtype: int64

In [64]:
df_frontdesk_rows['Redirecting number'].value_counts()

Redirecting number
13123411070      17145
13123411070.0    15054
13125068647        410
13125068647.0      300
13125068646        208
13125068646.0      138
13123478342.0      106
13123478342        104
13127536357.0       82
13124312299         28
13127536356.0       26
2302.0              12
13125068649.0       12
13124312299.0       12
13125068649          4
13122296080.0        4
13122296080          4
2302                 3
13122296013          2
13123478399          2
13127536356          2
13122296001.0        2
2303                 1
Name: count, dtype: int64

In [65]:
df_frontdesk_calls = df_useful.loc[df_useful['Correlation ID'].isin(df_frontdesk_rows['Correlation ID']),:].reset_index(drop = True)

In [66]:
df_frontdesk_calls.dtypes

Correlation ID                object
Called number                 object
PSTN vendor name              object
Direction                     object
Duration                       int64
Original reason               object
Redirect reason               object
Redirecting number            object
Start time            datetime64[ns]
User type                     object
User                          object
Answered                        bool
Related reason                object
dtype: object

In [67]:
df_frontdesk_calls.iloc[0:10,:]

,Correlation ID,Called number,PSTN vendor name,Direction,Duration,Original reason,Redirect reason,Redirecting number,Start time,User type,User,Answered,Related reason
0,00024966-393c-467b-ac36-77748d6dd2cf,13122296300,CallTower,TERMINATING,168,NaN,NaN,NaN,2025-02-04 08:17:48.124,User,NaN,True,Deflection
1,00024966-393c-467b-ac36-77748d6dd2cf,13123478312,NaN,TERMINATING,34,Deflection,Deflection,13123411070,2025-02-04 08:19:50.405,User,NaN,True,NaN
2,00024966-393c-467b-ac36-77748d6dd2cf,13123478312,NaN,ORIGINATING,34,Deflection,Deflection,13122296300,2025-02-04 08:19:50.405,User,NaN,True,Deflection
3,00024966-393c-467b-ac36-77748d6dd2cf,13123478300,NaN,ORIGINATING,34,Deflection,NoAnswer,13123478312,2025-02-04 08:20:08.583,User,NaN,True,CallForwardNoAnswer
4,00024966-393c-467b-ac36-77748d6dd2cf,13123478300,NaN,TERMINATING,34,Deflection,NoAnswer,13123411070,2025-02-04 08:20:08.583,VoiceMailRetrieval,NaN,True,NaN
5,0004eb97-96eb-4d42-90bc-0abd9101be7e,13122296300,NaN,TERMINATING,147,NaN,NaN,NaN,2024-04-10 06:15:06.352,User,NaN,True,ConsultativeTransfer
6,000da2e5-6855-4dbd-938d-f8764267364f,13123411070,CallTower,TERMINATING,59,NaN,NaN,NaN,2024-04-19 10:39:19.148,Unknown,NaN,True,NaN
7,000da2e5-6855-4dbd-938d-f8764267364f,13122296300,NaN,ORIGINATING,13,FollowMe,FollowMe,13123411070.0,2024-04-19 10:39:55.960,Unknown,NaN,True,NaN
8,000da2e5-6855-4dbd-938d-f8764267364f,13122296300,NaN,TERMINATING,13,FollowMe,FollowMe,13123411070.0,2024-04-19 10:39:55.960,User,NaN,True,NaN
9,0012b095-3d63-4ef4-a5c6-53482bc4f038,13123411070,CallTower,TERMINATING,5579,NaN,NaN,NaN,2025-08-05 08:10:21.934,WCCAdapter,NaN,True,NaN


In [68]:
adhoc_data_path = 'C:\\Users\\akl0407\\OneDrive - Northwestern University\\Back up\\2025-26\\Fall 2025\\LegalAid\\Adhoc\\Adhoc datasets\\'

In [69]:
df_frontdesk_calls.to_csv(adhoc_data_path + 'frontdesk.csv')

In [70]:
df_frontdesk_calls.groupby('Correlation ID').first()['Called number'].value_counts()

Called number
13123411070    15918
13122296300     2489
13123478342      105
1182              31
13124312299       19
1180              13
13125068646        4
13122296080        4
13125068647        3
13122296346        2
13122296013        1
17732099681        1
13122296342        1
13129561703        1
17734066091        1
13122296321        1
13122296001        1
Name: count, dtype: int64

In [11]:
15.6+2.3

17.9

In [10]:
17134+452

17586

In [79]:
df_main.head()

,Start time,Remote SessionID,Original reason,Route group,User type,Call outcome,Report ID,Outbound trunk,Related reason,Direction,...,Column1,PSTN vendor name2,User,Original reason2,Original called party UUID,Recall Type,Hold Duration,Auto Attendant Key Pressed,Queue Type,Answered Elsewhere
600224,2025-06-27 06:25:23.791,NaN,NaN,NaN,WCCAdapter,Success,b6dc159f-c2b6-3e07-a611-ded33f673ef2,wcc_Pc_tp-ipRwm_ku064NHZiw,NaN,TERMINATING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
929258,2024-10-05 07:30:09.903,NaN,NaN,NaN,WCCAdapter,Success,ec674d05-e84d-3bcc-bf05-3a148df3e1ef,wcc_Iyq3fhu9TjS8hbku8c4Zcg,NaN,TERMINATING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
267542,2024-12-11 08:41:47.355,NaN,NaN,NaN,User,Success,aacea7b9-5190-3aab-b568-a5295a04eb5e,NaN,NaN,TERMINATING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
267540,2024-12-11 08:42:05.358,41424040edde48e4838bffebf8492c67,NoAnswer,NaN,User,Success,0a882181-8a09-320c-8336-39dfc0bffed4,NaN,CallForwardNoAnswer,ORIGINATING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
267541,2024-12-11 08:42:05.358,NaN,NoAnswer,NaN,VoiceMailRetrieval,Success,5aa6b6a0-dd21-3be4-8edb-77e65f36d3d1,NaN,NaN,TERMINATING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
